# 5-Fold Grouped Cross-Validation

Robustness check for *Comparative Analysis of CNN and Vision Transformer Architectures
for Explainable Endometriosis Classification in Laparoscopic Images*.

The main results use one held-out, case-wise test split. A reviewer will ask whether
those numbers are a lucky split, so this notebook retrains **ResNet50, EfficientNet-B0
and ViT-B/16** across **5 grouped folds** over all 746 images and reports mean ± std.

**Rules that must not be broken**
- `StratifiedGroupKFold` over the 122 surgeries — never plain `KFold`.
  A frame from one surgery must never appear in two folds, or the result is leakage.
  GLENDA splits no-pathology recordings into per-segment folders, so the segment
  suffix is stripped before grouping.
- Each fold carves its **own validation set, also group-wise**, for early stopping.
  The fold's test part is never used for model selection.
- `seed = 42`, identical hyperparameters in every fold.
- **Per-image predictions are saved**, not just averages — significance tests need them.

**Before running:** Runtime → Change runtime type → **T4 GPU**. The dataset is
copied automatically from `MyDrive/endometriosis_paper`, and every result is
written back into that same folder, so nothing needs uploading by hand.

Expect roughly **1.5–2 hours** for all 15 runs (3 models × 5 folds). Results are written
after every single run, so a disconnect loses at most one run — just re-run the training
cell and it resumes.

In [1]:
# Everything lives in one Drive folder: the dataset goes in, the results come out.
# Change this single line if you rename or move the folder.
PROJECT_DIR = "/content/drive/MyDrive/endometriosis_paper"

import shutil
from pathlib import Path

ZIP = Path("glenda_balanced.zip")          # working copy inside the session
DRIVE_ZIP = Path(PROJECT_DIR) / "glenda_balanced.zip"

if not ZIP.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount failed:", exc)
    if DRIVE_ZIP.exists():
        print(f"Copying the dataset from {DRIVE_ZIP} ...")
        shutil.copy(DRIVE_ZIP, ZIP)
    else:
        raise SystemExit(
            f"glenda_balanced.zip is not in {PROJECT_DIR}. Put it there, "
            "or upload it into this session (left sidebar > Files > Upload)."
        )
print(f"dataset zip: {ZIP.stat().st_size / 1e6:.0f} MB")


Drive mount failed: Error: credential propagation was unsuccessful


SystemExit: glenda_balanced.zip is not in /content/drive/MyDrive/endometriosis_paper. Put it there, or upload it into this session (left sidebar > Files > Upload).

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
!unzip -q -o glenda_balanced.zip -d data
!ls data


## 1. Setup

If Drive mounts, results are written there so a Colab disconnect cannot lose them.

In [ ]:
import os, random, time, csv
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
N_FOLDS = 5
IMG, BATCH = 224, 32
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
MAX_EPOCHS, PATIENCE = 30, 6

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
# cuDNN picks algorithms by benchmarking, which makes runs differ by ~0.02 AUC;
# fixing it costs a little speed and buys a reproducible result
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if device.type != "cuda":
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, or this will take many hours.")

# every result is written into the same Drive folder the dataset came from,
# so a Colab disconnect cannot lose a finished fold
try:
    PROJECT_DIR
except NameError:                       # cell 1 was skipped
    PROJECT_DIR = "/content/drive/MyDrive/endometriosis_paper"

MOUNT_ROOT = Path("/content/drive/MyDrive")
if not MOUNT_ROOT.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount raised:", exc)

# mount() can fail WITHOUT raising ("credential propagation was unsuccessful").
# Never create the Drive path by hand: that would silently write results into the
# throwaway VM, and a disconnect would destroy them.
if MOUNT_ROOT.exists():
    OUT_DIR = Path(PROJECT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    OUT_DIR = Path("/content")
    print("=" * 72)
    print("WARNING: Google Drive is NOT mounted, results go to the session only.")
    print("A disconnect will destroy them, and the resume-after-disconnect feature")
    print("will not work either. To fix it: run the cell above again and complete")
    print("the permission popup (allow third-party cookies for colab, or try")
    print("another browser / an incognito-free window). You can also keep going and")
    print("download the CSVs from the last cell the moment the run finishes.")
    print("=" * 72)

PRED_CSV = OUT_DIR / "cv_predictions.csv"
FOLD_CSV = OUT_DIR / "cv_fold_metrics.csv"
print("Output dir:", OUT_DIR)

## 2. Pool every image and build the folds

The zip ships the original train/val/test directories. Cross-validation needs all 746
images in one pool, re-partitioned by group. `manifest.csv` already carries the class,
the label and the `group_id` (case id for pathology, video id for normals).

In [ ]:
DATA = Path("data")
manifest = pd.read_csv(DATA / "manifest.csv")
manifest["path"] = [str(DATA / s / c / f) for s, c, f in
                    zip(manifest["split"], manifest["class"], manifest["filename"])]
missing = [p for p in manifest["path"] if not Path(p).exists()]
assert not missing, f"{len(missing)} files from the manifest are missing, e.g. {missing[:3]}"

y = manifest["label"].to_numpy()            # 1 = endometriosis, 0 = normal

# GLENDA keeps no-pathology frames in per-segment folders, so one recording can
# appear as several group ids (v_4770_s_30-361, v_4770_s_2649-3117, ...). Two
# segments of one recording are the same surgery and must never land on opposite
# sides of a split, so the segment suffix is stripped. Pathology frames are
# already grouped by case id, which is stricter still.
manifest["surgery_id"] = manifest["group_id"].str.replace(r"_s_[-\d]+$", "",
                                                          regex=True)
groups = manifest["surgery_id"].to_numpy()
collapsed = manifest["group_id"].nunique() - manifest["surgery_id"].nunique()
print(f"grouping by surgery: {manifest['surgery_id'].nunique()} groups "
      f"({collapsed} segment folders merged back into their recording)")
print(f"{len(manifest)} images | {manifest['surgery_id'].nunique()} surgeries | "
      f"{int(y.sum())} endometriosis / {int((y == 0).sum())} normal")

outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = []
for fold, (tr_val_idx, test_idx) in enumerate(outer.split(manifest["path"], y, groups)):
    # inner group-wise split of the remaining data -> validation set for early stopping
    inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    tr_rel, val_rel = next(inner.split(tr_val_idx, y[tr_val_idx], groups[tr_val_idx]))
    train_idx, val_idx = tr_val_idx[tr_rel], tr_val_idx[val_rel]

    # leakage guards: no group may be shared between any two parts
    g_tr, g_va, g_te = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
    assert not (g_tr & g_va), "group leaks train<->val"
    assert not (g_tr & g_te), "group leaks train<->test"
    assert not (g_va & g_te), "group leaks val<->test"

    FOLDS.append({"train": train_idx, "val": val_idx, "test": test_idx})
    print(f"fold {fold}: train {len(train_idx):3d} ({int(y[train_idx].sum())} endo) | "
          f"val {len(val_idx):3d} ({int(y[val_idx].sum())} endo) | "
          f"test {len(test_idx):3d} ({int(y[test_idx].sum())} endo) | "
          f"groups {len(g_tr)}/{len(g_va)}/{len(g_te)}")

all_test = np.concatenate([f["test"] for f in FOLDS])
assert len(set(all_test)) == len(manifest), "the folds must cover every image exactly once"
print("\nLeakage checks passed; the 5 test parts tile the whole dataset.")

# A signature of the actual frame -> fold assignment. Results may only be resumed
# when this matches, so a change of grouping can never be silently mixed into an
# existing output file.
import hashlib

assignment = {}
for f, split in enumerate(FOLDS):
    for i in split["test"]:
        assignment[manifest["filename"].to_numpy()[i]] = f
FOLD_SIGNATURE = hashlib.sha256(
    "|".join(f"{k}:{v}" for k, v in sorted(assignment.items())).encode()
).hexdigest()[:12]
print("fold signature:", FOLD_SIGNATURE)

## 3. Dataset, models and the training loop

Same recipe as the main experiment: ImageNet weights, full fine-tuning, Adam,
light augmentation on train only, early stopping on **validation AUC**.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


class FrameDataset(Dataset):
    """Indexes the pooled manifest, so folds are plain index arrays."""

    def __init__(self, indices, train: bool):
        self.paths = manifest["path"].to_numpy()[indices]
        self.labels = y[indices]
        self.tf = train_tf if train else eval_tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), int(self.labels[i])


def build_resnet50():
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m


def build_efficientnet_b0():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    return m


def build_vit_b16():
    m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    m.heads.head = nn.Linear(m.heads.head.in_features, 2)
    return m


MODELS = {                      # name -> (builder, learning rate)
    "ResNet50": (build_resnet50, 1e-4),
    "EfficientNet-B0": (build_efficientnet_b0, 1e-4),
    "ViT-B/16": (build_vit_b16, 2e-5),
}
POS = 1                         # column 1 of the softmax = endometriosis


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, trues = [], []
    for x, t in loader:
        p = torch.softmax(model(x.to(device)), 1)[:, POS].cpu().numpy()
        probs += p.tolist(); trues += t.numpy().tolist()
    return np.asarray(probs), np.asarray(trues)


def train_one(model, lr, train_loader, val_loader, tag):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    best_auc, best_state, wait = 0.0, None, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        for x, t in train_loader:
            x, t = x.to(device), t.to(device)
            opt.zero_grad(); crit(model(x), t).backward(); opt.step()
        val_prob, val_true = predict(model, val_loader)
        auc = roc_auc_score(val_true, val_prob)
        if auc > best_auc:
            best_auc, wait = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"    {tag}: early stop @ epoch {ep + 1} (best val AUC {best_auc:.3f})")
                break
    model.load_state_dict(best_state)
    return model, best_auc

## 3b. Explanation scoring, inside the fold loop

The first version of this notebook discarded each model after scoring it, so the
Grad-CAM analysis had to borrow weights from an earlier single-split experiment
and covered only 53 frames. Scoring the maps here - while the fold's model is
still in memory - covers **all 373 pathology frames** with the very models that
produce the classification numbers, and costs a few seconds per fold.


In [ ]:
!pip install -q grad-cam

from contextlib import contextmanager

from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

CAM_THRESHOLD = 0.5
GRID = 14                      # 224 / 16 patches per side
N_TOKENS = GRID * GRID + 1
MASK_DIRS = [DATA / 'masks' / s / 'endometriosis' for s in ('train', 'val', 'test')]


def mask_path(filename):
    """The zip keeps masks under the original split folders; the folds ignore
    those folders, so look the mask up by name."""
    stem = Path(filename).stem + '.png'
    for d in MASK_DIRS:
        p = d / stem
        if p.exists():
            return p
    return None


def load_mask(path):
    m = np.array(Image.open(path).convert('L').resize((224, 224)))
    return (m > 0).astype(np.uint8)


def preprocess(path):
    img = Image.open(path).convert('RGB').resize((224, 224))
    arr = np.array(img) / 255.0
    x = torch.tensor(arr).permute(2, 0, 1).float()
    x = (x - torch.tensor(MEAN).view(3, 1, 1)) / torch.tensor(STD).view(3, 1, 1)
    return x.unsqueeze(0), arr


def iou(cam, mask, thr=CAM_THRESHOLD):
    pred = (cam >= thr).astype(np.uint8)
    union = (pred | mask).sum()
    return float((pred & mask).sum() / union) if union else 0.0


def pointing_and_coverage(cam, mask):
    peak = np.unravel_index(np.argmax(cam), cam.shape)
    return int(mask[peak] > 0), float((cam * mask).sum() / (cam.sum() + 1e-8))


def vit_reshape(t, h=GRID, w=GRID):
    r = t[:, 1:, :].reshape(t.size(0), h, w, t.size(2))
    return r.transpose(2, 3).transpose(1, 2)


def target_layers(name, model):
    if name == 'ResNet50':
        return [model.layer4[-1]], None
    if name == 'EfficientNet-B0':
        return [model.features[-1]], None
    return [model.encoder.layers[-1].ln_1], vit_reshape


@contextmanager
def capture_attention(model):
    """torchvision's EncoderBlock calls MultiheadAttention with
    need_weights=False, so the weights have to be asked for explicitly."""
    store, originals = [], []
    for block in model.encoder.layers:
        attn = block.self_attention
        originals.append((attn, attn.forward))

        def patched(query, key, value, _attn=attn, **kwargs):
            kwargs.pop('need_weights', None)
            kwargs.pop('average_attn_weights', None)
            out, weights = type(_attn).forward(
                _attn, query, key, value,
                need_weights=True, average_attn_weights=True, **kwargs)
            store.append(weights.detach())
            return out, weights

        attn.forward = patched
    try:
        yield store
    finally:
        for attn, original in originals:
            attn.forward = original


def rollout(attentions):
    """Abnar & Zuidema: add the residual, row-normalise, multiply the layers."""
    eye = torch.eye(N_TOKENS)
    result = eye.clone()
    for attn in attentions:
        a = attn[0].cpu() + eye
        a = a / a.sum(dim=-1, keepdim=True)
        result = a @ result
    grid = result[0, 1:].reshape(1, 1, GRID, GRID)
    up = torch.nn.functional.interpolate(grid, size=(224, 224), mode='bilinear',
                                         align_corners=False)
    cam = up[0, 0].numpy()
    span = cam.max() - cam.min()
    return (cam - cam.min()) / span if span > 0 else np.zeros_like(cam)


def localize(model, name, fold, test_idx):
    """Score this fold's pathology frames while the model is still in memory."""
    rows = []
    paths = manifest['path'].to_numpy()
    names = manifest['filename'].to_numpy()
    groups_arr = manifest['surgery_id'].to_numpy()
    layers, reshape = target_layers(name, model)
    cam_engine = GradCAM(model=model, target_layers=layers, reshape_transform=reshape)
    for i in test_idx:
        if y[i] != 1:                      # masks exist only for pathology frames
            continue
        mpath = mask_path(names[i])
        if mpath is None:
            continue
        x, _ = preprocess(paths[i])
        x = x.to(device)
        mask = load_mask(mpath)

        cam = cam_engine(input_tensor=x, targets=[ClassifierOutputTarget(POS)])[0]
        hit, cov = pointing_and_coverage(cam, mask)
        rows.append([name, 'grad-cam', fold, names[i], groups_arr[i],
                     f'{iou(cam, mask):.6f}', hit, f'{cov:.6f}',
                     f'{float((cam >= CAM_THRESHOLD).mean()):.6f}',
                     f'{float(mask.mean()):.6f}'])

        if name == 'ViT-B/16':             # the transformer's own explanation
            with capture_attention(model) as store:
                store.clear()
                with torch.no_grad():
                    model(x)
                cam_r = rollout(store)
            hit_r, cov_r = pointing_and_coverage(cam_r, mask)
            rows.append([name, 'rollout', fold, names[i], groups_arr[i],
                         f'{iou(cam_r, mask):.6f}', hit_r, f'{cov_r:.6f}',
                         f'{float((cam_r >= CAM_THRESHOLD).mean()):.6f}',
                         f'{float(mask.mean()):.6f}'])
    return rows


print('localization helpers ready')


## 4. Run the 15 fits

Each `(model, fold)` result is appended to `cv_predictions.csv` as soon as it finishes.
Re-running this cell skips whatever is already in that file, so a Colab disconnect costs
at most one run.

In [ ]:
def done_runs():
    """Finished (model, fold) pairs - but only if they were computed under the
    fold assignment this session built."""
    if not PRED_CSV.exists():
        return set()
    if not SIG_FILE.exists() or SIG_FILE.read_text().strip() != FOLD_SIGNATURE:
        stamp = time.strftime("%Y%m%d-%H%M%S")
        for p in (PRED_CSV, FOLD_CSV, LOC_CSV):
            if p.exists():
                p.rename(p.with_name(f"{p.stem}.superseded-{stamp}.csv"))
        print("The existing results were produced under a DIFFERENT fold assignment.")
        print("They have been renamed *.superseded-*.csv and this run starts clean.")
        SIG_FILE.write_text(FOLD_SIGNATURE)
        return set()
    seen = pd.read_csv(PRED_CSV)
    return set(zip(seen["model"], seen["fold"]))


def append_rows(path, rows, header):
    exists = path.exists()
    with open(path, "a", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        if not exists:
            w.writerow(header)
        w.writerows(rows)


# this run must redo every fold: the first run saved no explanation scores,
# and reusing its predictions with freshly trained models would mix two sets of
# weights in one analysis
PRED_CSV = OUT_DIR / "cv_predictions_v3.csv"
FOLD_CSV = OUT_DIR / "cv_fold_metrics_v3.csv"
LOC_CSV = OUT_DIR / "cv_localization_v3.csv"
SIG_FILE = OUT_DIR / "cv_fold_signature_v3.txt"
if not SIG_FILE.exists():
    SIG_FILE.write_text(FOLD_SIGNATURE)
print("writing to:", PRED_CSV.name, FOLD_CSV.name, LOC_CSV.name)

already = done_runs()
if already:
    print(f"Resuming: {len(already)} of {len(MODELS) * N_FOLDS} runs already finished")

for name, (builder, lr) in MODELS.items():
    for fold, split in enumerate(FOLDS):
        if (name, fold) in already:
            print(f"[{name} fold {fold}] skipped (already done)")
            continue
        t0 = time.time()
        # fresh seed per run so folds are reproducible independently of order
        torch.manual_seed(SEED + fold); torch.cuda.manual_seed_all(SEED + fold)
        np.random.seed(SEED + fold); random.seed(SEED + fold)

        train_loader = DataLoader(FrameDataset(split["train"], True), BATCH,
                                  shuffle=True, num_workers=2, drop_last=False)
        val_loader = DataLoader(FrameDataset(split["val"], False), BATCH,
                                shuffle=False, num_workers=2)
        test_loader = DataLoader(FrameDataset(split["test"], False), BATCH,
                                 shuffle=False, num_workers=2)

        print(f"[{name} fold {fold}] training on {len(split['train'])} images...")
        model, val_auc = train_one(builder(), lr, train_loader, val_loader, f"{name} fold {fold}")

        prob, true = predict(model, test_loader)
        pred = (prob >= 0.5).astype(int)
        metrics = {
            "Accuracy": accuracy_score(true, pred),
            "Precision": precision_score(true, pred, zero_division=0),
            "Recall": recall_score(true, pred, zero_division=0),
            "F1": f1_score(true, pred, zero_division=0),
            "AUC": roc_auc_score(true, prob),
        }
        print(f"    done in {time.time() - t0:.0f}s | test AUC {metrics['AUC']:.3f} "
              f"acc {metrics['Accuracy']:.3f} recall {metrics['Recall']:.3f}")

        idx = split["test"]
        append_rows(PRED_CSV,
                    [[name, fold, manifest["filename"].to_numpy()[i],
                      manifest["surgery_id"].to_numpy()[i],
                      manifest["group_id"].to_numpy()[i],
                      int(y[i]), f"{p:.6f}", int(p >= 0.5)]
                     for i, p in zip(idx, prob)],
                    ["model", "fold", "filename", "surgery_id", "segment_id",
                     "y_true", "prob", "pred"])
        append_rows(FOLD_CSV,
                    [[name, fold, len(split["train"]), len(split["test"]), round(val_auc, 4)]
                     + [round(metrics[k], 4) for k in
                        ("Accuracy", "Precision", "Recall", "F1", "AUC")]],
                    ["model", "fold", "n_train", "n_test", "val_AUC",
                     "Accuracy", "Precision", "Recall", "F1", "AUC"])
        loc_rows = localize(model, name, fold, idx)
        append_rows(LOC_CSV, loc_rows,
                    ["model", "method", "fold", "filename", "surgery_id",
                     "IoU", "pointing", "coverage", "area", "mask_area"])
        print(f"    localization: {len(loc_rows)} rows "
              f"({sum(1 for r in loc_rows if r[1] == 'grad-cam')} pathology frames)")

        del model
        torch.cuda.empty_cache()

print("\nAll runs complete.")

## 5. Aggregate

Two numbers matter per model:

- **mean ± std across the 5 folds** — the robustness claim for the paper
- **pooled out-of-fold AUC** — every one of the 746 images predicted by a model that
  never saw its group, which is a single, tighter estimate than any one fold

In [ ]:
folds = pd.read_csv(FOLD_CSV).drop_duplicates(subset=["model", "fold"], keep="last")
preds = pd.read_csv(PRED_CSV).drop_duplicates(subset=["model", "fold", "filename"], keep="last")

metric_cols = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
summary = folds.groupby("model")[metric_cols].agg(["mean", "std"]).round(4)
print("=== 5-fold cross-validation (mean +- std) ===")
for name in MODELS:
    if name not in summary.index:
        continue
    row = summary.loc[name]
    print(f"{name:16s} " + "  ".join(
        f"{m} {row[(m, 'mean')]:.3f}+-{row[(m, 'std')]:.3f}" for m in metric_cols))

print("\n=== Pooled out-of-fold (all 746 images) ===")
pooled_rows = []
for name in MODELS:
    sub = preds[preds["model"] == name]
    if sub.empty:
        continue
    auc = roc_auc_score(sub["y_true"], sub["prob"])
    acc = accuracy_score(sub["y_true"], sub["pred"])
    rec = recall_score(sub["y_true"], sub["pred"], zero_division=0)
    prec = precision_score(sub["y_true"], sub["pred"], zero_division=0)
    f1 = f1_score(sub["y_true"], sub["pred"], zero_division=0)
    pooled_rows.append([name, len(sub), round(acc, 4), round(prec, 4), round(rec, 4),
                        round(f1, 4), round(auc, 4)])
    print(f"{name:16s} n={len(sub)}  AUC {auc:.3f}  acc {acc:.3f}  recall {rec:.3f}")

summary.to_csv(OUT_DIR / "cv_summary.csv")
pd.DataFrame(pooled_rows, columns=["model", "n", "Accuracy", "Precision", "Recall", "F1", "AUC"]
             ).to_csv(OUT_DIR / "cv_pooled.csv", index=False)
print("\nSaved cv_summary.csv and cv_pooled.csv in", OUT_DIR)
# --- localization, pooled over every fold's pathology frames ---
loc = pd.read_csv(LOC_CSV).drop_duplicates(
    subset=["model", "method", "fold", "filename"], keep="last")
print(f"\n=== Grad-CAM / rollout vs the expert masks "
      f"({loc['filename'].nunique()} frames, {loc['surgery_id'].nunique()} surgeries) ===")
for (m, meth), g in loc.groupby(["model", "method"]):
    print(f"{m:16s} {meth:9s} IoU {g['IoU'].mean():.4f}  "
          f"Pointing {g['pointing'].mean():.4f}  Coverage {g['coverage'].mean():.4f}  "
          f"(n={len(g)})")
loc.to_csv(LOC_CSV, index=False)


## 6. Optional: download the four CSVs

They are already in `PROJECT_DIR` on Drive. This cell only makes browser copies as well; put them in the repo's `results/` folder when convenient.

In [ ]:
# The four CSVs are already saved in PROJECT_DIR on your Drive, next to the
# weights and the earlier results. Run this only if you also want them
# downloaded straight into your browser's Downloads folder.
from google.colab import files

for f in ["cv_predictions_v3.csv", "cv_fold_metrics_v3.csv", "cv_localization_v3.csv",
          "cv_summary.csv", "cv_pooled.csv"]:
    p = OUT_DIR / f
    if p.exists():
        files.download(str(p))
    else:
        print("missing:", p)
